# Iceberg Basics — 02: Reading and Writing

## What you will learn
Iceberg supports multiple write APIs — SQL, DataFrame writeTo, and format("iceberg").
Each has different strengths for different use cases.

In this notebook:
1. Write APIs — SQL INSERT, writeTo, format("iceberg")
2. Read APIs — spark.table vs format("iceberg")
3. Append vs overwrite vs dynamic overwrite
4. `fanout-enabled` — writing to multiple partitions safely
5. Reading with filters — partition pruning
6. Write-back pattern — read, transform, write


In [1]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:27:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Iceberg catalog ready


26/04/10 20:27:40 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:27:42 WARN TaskSetManager: Stage 1 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


In [2]:
spark.sql("DROP TABLE IF EXISTS local.icedb.rw_orders")

# Create Iceberg table from DataFrame using writeTo API
df.writeTo("local.icedb.rw_orders") \
  .partitionedBy("category") \
  .using("iceberg") \
  .createOrReplace()
print(f"Table created: {spark.table('local.icedb.rw_orders').count():,} rows")

26/04/10 20:27:43 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:27:45 WARN HadoopTableOperations: Error reading version hint file /workspace/data/warehouse/iceberg/icedb/rw_orders/metadata/version-hint.text
java.io.FileNotFoundException: File /workspace/data/warehouse/iceberg/icedb/rw_orders/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:189)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:572)
	at org.apache.hadoop.fs.Fil

Table created: 100,000 rows


## Step 1 — Write APIs Compared


In [3]:
# API 1: SQL INSERT INTO
spark.createDataFrame(
    [(200001, 1, "Laptop", "Electronics", "US", 1, 999.99, 999.99, "confirmed",
      datetime.date(2024,7,1))], df.schema
).createOrReplaceTempView("one_row")

spark.sql("INSERT INTO local.icedb.rw_orders SELECT * FROM one_row")
print(f"API 1 SQL INSERT: {spark.table('local.icedb.rw_orders').count():,} rows")

# API 2: writeTo().append()
extra = df.limit(500)
extra.writeTo("local.icedb.rw_orders").append()
print(f"API 2 writeTo.append: {spark.table('local.icedb.rw_orders').count():,} rows")

# API 3: format("iceberg") — path-based
import pathlib
warehouse = "/workspace/data/warehouse/iceberg"
table_path = f"{warehouse}/icedb/rw_orders"
extra.write.format("iceberg").mode("append").save(table_path)
print(f"API 3 format iceberg: {spark.table('local.icedb.rw_orders').count():,} rows")

API 1 SQL INSERT: 100,001 rows


26/04/10 20:27:47 WARN TaskSetManager: Stage 16 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


API 2 writeTo.append: 100,501 rows


26/04/10 20:27:48 WARN TaskSetManager: Stage 25 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


API 3 format iceberg: 100,501 rows


## Step 2 — overwritePartitions: Dynamic Partition Overwrite


In [4]:
# Replace only specific partitions — not entire table
before_elec = spark.table("local.icedb.rw_orders") \
                   .filter(F.col("category")=="Electronics").count()
before_total = spark.table("local.icedb.rw_orders").count()

# New Electronics data (reprocessed)
new_elec = df.filter(F.col("category")=="Electronics") \
             .withColumn("price", F.col("price") * 0.9)  # 10% price drop

# overwritePartitions replaces only partitions present in new data
new_elec.writeTo("local.icedb.rw_orders").overwritePartitions()

after_elec  = spark.table("local.icedb.rw_orders").filter(F.col("category")=="Electronics").count()
after_total = spark.table("local.icedb.rw_orders").count()

print(f"Dynamic partition overwrite (Electronics only):")
print(f"  Before: total={before_total:,}  electronics={before_elec:,}")
print(f"  After : total={after_total:,}  electronics={after_elec:,}")
print(f"  Other categories: unchanged ✅")

26/04/10 20:27:50 WARN TaskSetManager: Stage 40 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


Dynamic partition overwrite (Electronics only):
  Before: total=100,501  electronics=16,578
  After : total=100,838  electronics=16,496
  Other categories: unchanged ✅


## Step 3 — Read APIs and Partition Pruning


In [5]:
# Read via catalog name (most readable)
df1 = spark.table("local.icedb.rw_orders")
print(f"spark.table(): {df1.count():,} rows")

# Read via format + path
df2 = spark.read.format("iceberg").load(f"{warehouse}/icedb/rw_orders")
print(f"format('iceberg'): {df2.count():,} rows")

# Partition pruning — filter on partition column
runs = 3
t_full, t_pruned = [], []
for _ in range(runs):
    t0=time.time(); spark.table("local.icedb.rw_orders").agg(F.sum("revenue")).collect(); t_full.append(time.time()-t0)
    t0=time.time(); spark.table("local.icedb.rw_orders").filter(F.col("category")=="Books").agg(F.sum("revenue")).collect(); t_pruned.append(time.time()-t0)

print(f"\nPartition pruning (category='Books'):")
print(f"  Full scan : {sorted(t_full)[1]:.3f}s")
print(f"  Pruned    : {sorted(t_pruned)[1]:.3f}s  (only Books partition)")
print(f"  Speedup   : {sorted(t_full)[1]/sorted(t_pruned)[1]:.1f}x")

spark.table(): 100,838 rows
format('iceberg'): 100,838 rows

Partition pruning (category='Books'):
  Full scan : 0.307s
  Pruned    : 0.354s  (only Books partition)
  Speedup   : 0.9x


## Step 4 — fanout-enabled for Parallel Partition Writes


In [6]:
# Without fanout — default: Spark routes rows to correct task before writing
# With fanout — each task can write to ALL partitions (needed for streaming/unsorted data)

print("fanout-enabled: allow tasks to write to multiple partitions")
print("Required when input data is NOT sorted by partition column")
print()
print("Usage:")
print("  df.write.format('iceberg')")
print("    .option('fanout-enabled', 'true')")
print("    .mode('append').save(path)")
print()
print("In streaming:")
print("  .writeStream.format('iceberg')")
print("    .option('fanout-enabled', 'true')")
print("    .toTable('catalog.db.table')")
print()
print("When to use: always for streaming, and for batch when data is not pre-sorted")

# Demo: write unsorted data with fanout
unsorted = df.orderBy(F.rand(42))  # random order
unsorted.write.format("iceberg") \
        .option("fanout-enabled", "true") \
        .mode("append") \
        .save(f"{warehouse}/icedb/rw_orders")
print(f"\nfanout write succeeded: {spark.table('local.icedb.rw_orders').count():,} rows")

fanout-enabled: allow tasks to write to multiple partitions
Required when input data is NOT sorted by partition column

Usage:
  df.write.format('iceberg')
    .option('fanout-enabled', 'true')
    .mode('append').save(path)

In streaming:
  .writeStream.format('iceberg')
    .option('fanout-enabled', 'true')
    .toTable('catalog.db.table')

When to use: always for streaming, and for batch when data is not pre-sorted


26/04/10 20:27:54 WARN TaskSetManager: Stage 73 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:27:54 WARN TaskSetManager: Stage 74 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.



fanout write succeeded: 100,838 rows


## Summary

```python
# Preferred: catalog-based writeTo API
df.writeTo("local.db.table").append()
df.writeTo("local.db.table").overwritePartitions()  # dynamic partition overwrite
df.writeTo("local.db.table").createOrReplace()      # replace entire table

# Path-based (when catalog not available)
df.write.format("iceberg").mode("append").save("/path/to/table")

# SQL
spark.sql("INSERT INTO local.db.table SELECT ...")

# Reading
spark.table("local.db.table")
spark.read.format("iceberg").load("/path/to/table")
spark.read.option("snapshot-id", snap_id).table("local.db.table")
```
